# α-AGI Insight · Discovery Workbench

Version 1.15.0. Five synthetic cases, exact review portfolios and reproducible evidence. No API key, provider calls, wallet or package installation required. Priority is not a forecast probability; every selected thesis still requires independent review. [Project notice](https://github.com/MontrealAI/AGI-Alpha-Agent-v0/blob/v1.15.0/docs/DISCLAIMER_SNIPPET.md).

Run all cells in order. Set `ALPHA_INSIGHT_SOURCE` to a local checkout for an offline run; otherwise the first cell clones the pinned release tag. Existing credentials are never modified.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Use Python 3.11–3.13")
local = os.environ.get("ALPHA_INSIGHT_SOURCE")
source = Path(local).expanduser().resolve() if local else Path("AGI-Alpha-Agent-v0-insight-1.15.0").resolve()
if not local:
    if not source.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "v1.15.0",
                        "https://github.com/MontrealAI/AGI-Alpha-Agent-v0.git", str(source)], check=True)
    head = subprocess.check_output(["git", "-C", str(source), "rev-parse", "HEAD"], text=True).strip()
    tag = subprocess.check_output(["git", "-C", str(source), "rev-parse", "v1.15.0^{commit}"], text=True).strip()
    if head != tag or subprocess.check_output(["git", "-C", str(source), "status", "--porcelain"], text=True):
        raise RuntimeError("Pinned checkout is modified; inspect it or choose a local ALPHA_INSIGHT_SOURCE")
if not (source / "alpha_factory_v1/demos/alpha_agi_insight_v0/discovery.py").is_file():
    raise RuntimeError("ALPHA_INSIGHT_SOURCE must point to the repository root")
sys.path.insert(0, str(source))
print("Source:", source)


## Compare all five cases

Review capacity, supplied source coverage and conservative priority determine the portfolio. All starter inputs are authored synthetic fixtures.

In [ ]:
from alpha_factory_v1.demos.alpha_agi_insight_v0.discovery import evaluate, read_json, verify, write_bundle

cases = read_json(source / "alpha_factory_v1/demos/alpha_agi_insight_v0/scenarios.json")
reports = [evaluate(case) for case in cases]
for report in reports:
    assert verify(report) == report
    result = report["result"]
    print(report["input"]["id"], result["status"], result["selectedIds"], result["reviewMinutesUsed"])


## Change an assumption

Set review capacity to zero: no opportunity can be selected, but its verification job remains available. Nothing is submitted or funded.

In [ ]:
from copy import deepcopy
scenario = deepcopy(cases[0])
scenario["policy"]["reviewMinutes"] = 0
empty = evaluate(scenario)
assert empty["result"]["selectedIds"] == []
assert empty["result"]["novaSeedDrafts"] == []
print(empty["result"]["status"])


## Retain a portable review bundle

Outputs are content-addressed. The six files bind the supplied scenario, dossier, unsubmitted jobs and plaintext unminted seed drafts. An exact rerun reuses its output; altered or partial output fails visibly.

In [ ]:
import zipfile
output = Path("insight-notebook-runs")
folder = write_bundle(reports[0], output)
assert verify(read_json(folder / "dossier.json")) == reports[0]
archive = output / (folder.name + ".zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(folder.iterdir()):
        bundle.write(path, path.name)
print("Bundle:", archive.resolve())


## Original search illustration

The numeric-target UCB example remains available. Sector labels map from numeric policies; these simulated rewards are not industry forecasts.

In [ ]:
import json
from alpha_factory_v1.demos.alpha_agi_insight_v0.insight_demo import run
legacy = json.loads(run(episodes=20, seed=42, offline=True, json_output=True))
print(legacy["scope"])
print(legacy["ranking"][:3])


## Next: independent verification

Review every source and success metric before using the job drafts. With the separately installed chain extra, `alpha-agent ascension-compile <folder>/jobs.json --output fusion-plan.json` prepares a plan for the independently operated Ascension lifecycle. This notebook does not mint, encrypt, escrow, trade or approve execution. The complete original notebook is preserved as `research_notebook_archive.ipynb`.